# Electrodynamics from Scratch — Lecture 9: The Magnetic Dipole Field

In Lecture 6 we computed the magnetic field of an arbitrary current loop with the Biot–Savart law,

$$\mathbf{B}(\mathbf{r}) = \frac{\mu_0 I}{4\pi} \oint \frac{d\boldsymbol{\ell} \times (\mathbf{r} - \mathbf{r}')}{|\mathbf{r} - \mathbf{r}'|^3}.$$

Far away from the loop — at distances $r$ much larger than the loop size $a$ — all the geometric detail of the wiring collapses into a single vector, the **magnetic dipole moment** $\mathbf{m} = I \mathbf{A}$ (current times the area vector, pointing by the right-hand rule). In that limit the exact integral reduces to the **point-dipole field**

$$\mathbf{B}(\mathbf{r}) = \frac{\mu_0}{4\pi} \frac{3 \hat{\mathbf{r}} (\mathbf{m} \cdot \hat{\mathbf{r}}) - \mathbf{m}}{r^3},$$

which falls off as $1/r^3$ — one power of $r$ faster than the electric field of a point charge, because a magnetic dipole has no magnetic "charge" to supply a $1/r^2$ term. On the dipole axis the field has magnitude $B = \mu_0\, 2m/(4\pi r^3)$, twice the equatorial value $B = \mu_0\, m/(4\pi r^3)$, where it points antiparallel to $\mathbf{m}$.

The dipole approximation is not merely a convenience: it is the leading term of an expansion in powers of $a/r$, the same multipole logic we used for charge distributions in Lecture 7. The first neglected correction for a loop scales like $(a/r)^2$, so the approximation improves rapidly as we back away from the source. A dipole also has a characteristic dynamics in an external field: the torque $\boldsymbol{\tau} = \mathbf{m} \times \mathbf{B}$ tries to align $\mathbf{m}$ with $\mathbf{B}$, and the associated potential energy $U = -\mathbf{m} \cdot \mathbf{B}$ is minimized in that aligned state — the physics behind compass needles and, at the atomic scale, the alignment of electron magnetic moments in a magnet.

In this lecture we verify the $1/r^3$ law numerically, draw the dipole field lines in the xz plane, quantify how fast the exact loop field converges to the dipole form, and check the torque formula against a direct segment-by-segment force calculation.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import electrodynamics_from_scratch as eds

## The $1/r^3$ law of the point dipole

We place a point dipole $\mathbf{m} = m_0 \hat{\mathbf{z}}$ at the origin and evaluate `eds.magnetic_dipole_field` along the $z$ axis (where $\mathbf{m} \parallel \hat{\mathbf{r}}$) and along the $x$ axis (the equatorial plane, where $\mathbf{m} \cdot \hat{\mathbf{r}} = 0$). The code checks the analytic magnitudes $2 k m_0/r^3$ and $k m_0/r^3$ with $k = \mu_0/(4\pi)$, fits the measured decay with `eds.power_law_fit`, and plots both curves against the exact $1/r^3$ reference.

In [ ]:
m_dip = np.array([0.0, 0.0, 1.0])          # dipole moment, A m^2
k_m = eds.MU_0 / (4.0 * np.pi)             # mu0 / (4 pi)

r = np.linspace(0.5, 3.0, 40)
axis_pts = np.column_stack([np.zeros_like(r), np.zeros_like(r), r])
eq_pts = np.column_stack([r, np.zeros_like(r), np.zeros_like(r)])

B_axis = eds.magnetic_dipole_field(m_dip, axis_pts)
B_eq = eds.magnetic_dipole_field(m_dip, eq_pts)

# On the axis B is along z with magnitude 2 k m0 / r^3;
# in the equatorial plane B is along -x with magnitude k m0 / r^3.
err_axis = np.max(np.abs(B_axis[:, 2] - 2.0 * k_m / r**3))
err_eq = np.max(np.abs(B_eq[:, 0] + k_m / r**3))
print("max |B_axis - 2 k m0/r^3| = %.3e T" % err_axis)
print("max |B_eq - (-k m0/r^3)|  = %.3e T" % err_eq)

amp, expo = eds.power_law_fit(r, np.abs(B_axis[:, 2]))
print("power-law fit on axis: |B| = %.4g * r^%.4f" % (amp, expo))
print("expected: amplitude %.4g, exponent -3" % (2.0 * k_m))

fig, ax = eds.plot_decay(
    r,
    {
        "axis |B|": np.abs(B_axis[:, 2]),
        "equator |B|": np.abs(B_eq[:, 0]),
        r"$2 k m_0 / r^3$": 2.0 * k_m / r**3,
    },
    title=r"Point dipole field: $1/r^3$ decay on axis and in the equatorial plane",
)
fig

## Field lines of the dipole in the xz plane

For $\mathbf{m} = m_0 \hat{\mathbf{z}}$ the field is axisymmetric: every plane containing the $z$ axis shows the same picture, so the xz plane captures the full structure. We build a `field_func(x, y)` that maps the plot mesh to 3D points $(x, 0, y)$ and returns the $(B_x, B_z)$ components, masking a small hole around the singular origin. The lines emerge from the "north" side, loop around, and re-enter at the "south" — closed loops, exactly as $\nabla \cdot \mathbf{B} = 0$ demands.

In [ ]:
CUT = 0.12   # mask radius around the singular origin

def dipole_b_xz(x, y):
    """(Bx, Bz) of the dipole on the xz-plane mesh (plot 'y' is z)."""
    pts = np.column_stack([x.ravel(), np.zeros(x.size), y.ravel()])
    B = eds.magnetic_dipole_field(m_dip, pts)
    r_pt = np.linalg.norm(pts, axis=1)
    B[r_pt < CUT] = 0.0
    return B[:, 0].reshape(x.shape), B[:, 2].reshape(x.shape)

fig, ax = eds.plot_field_lines(
    dipole_b_xz,
    extent=2.0,
    density=1.4,
    title=r"Field lines of a point dipole (xz plane, $\\mathbf{m} = m_0 \\hat{\\mathbf{z}}$)",
)
fig

## Exact loop field vs the dipole approximation

Now the key comparison: a circular loop of radius $a = 0.1$ m carrying $I = 1$ A, whose exact field we get from `eds.biot_savart` with 200 segments, versus `eds.magnetic_dipole_field` with the loop's exact moment $m = I \pi a^2$. We sample on the axis at distances from $1.5a$ to $20a$ and track the relative error, which should fall like $(a/r)^2$ — the hallmark of a multipole expansion truncated after the dipole term.

In [ ]:
a = 0.1          # loop radius, m
I_loop = 1.0     # current, A
loop = eds.circular_loop_curve(a, np.zeros(3), n=200)
m_loop = I_loop * np.pi * a**2 * np.array([0.0, 0.0, 1.0])

z = np.linspace(1.5 * a, 20.0 * a, 25)   # on-axis distances
pts = np.column_stack([np.zeros_like(z), np.zeros_like(z), z])

B_exact = eds.biot_savart(loop, I_loop, pts, closed=True)
B_dip = eds.magnetic_dipole_field(m_loop, pts)

rel = np.linalg.norm(B_exact - B_dip, axis=1) / np.linalg.norm(B_exact, axis=1)
amp, expo = eds.power_law_fit(z, rel)
print("error fit: relative error = %.4g * (r/m)^%.4f" % (amp, expo))
print("expected exponent: -2 (the next multipole term scales as (a/r)^2)")

# (a/r)^2 reference curve, scaled to pass through the first data point.
ref = rel[0] * (z[0] / a)**2 * (a / z)**2

fig1, ax1 = eds.plot_decay(
    z,
    {
        "exact loop |B|": np.abs(B_exact[:, 2]),
        "dipole approx |B|": np.abs(B_dip[:, 2]),
    },
    title="Loop field vs dipole approximation (on axis)",
)
fig2, ax2 = eds.plot_decay(
    z,
    {
        "relative error": rel,
        r"$(a/r)^2$ reference": ref,
    },
    title=r"Dipole-approximation error falls like $(a/r)^2$",
)
fig2

## The dipole moment and the torque in a uniform field

First we confirm that `eds.magnetic_dipole_moment` returns $\mathbf{m} = I A \hat{\mathbf{z}}$ for the loop. Then, in a uniform field $\mathbf{B}_0 = B_0 \hat{\mathbf{x}}$, we compute the torque two independent ways: numerically, by summing $\mathbf{r} \times d\mathbf{F}$ over the loop segments with $d\mathbf{F} = I\, d\boldsymbol{\ell} \times \mathbf{B}_0$, and analytically via $\boldsymbol{\tau} = \mathbf{m} \times \mathbf{B}_0$. The two must agree — that agreement is exactly why the dipole moment is a useful abstraction. Finally we plot $|\tau| = mB\sin\theta$ and $U = -\mathbf{m} \cdot \mathbf{B} = -mB\cos\theta$ as functions of the tilt angle.

In [ ]:
m_check = eds.magnetic_dipole_moment(loop, I_loop, closed=True)
m_expected = I_loop * np.pi * a**2 * np.array([0.0, 0.0, 1.0])
print("m from package :", m_check)
print("m = I A z-hat  :", m_expected)
print("agree:", np.allclose(m_check, m_expected))

B0 = np.array([1.0, 0.0, 0.0])   # uniform field along x

# Segment-by-segment forces dF = I dl x B0 on the closed loop.
start = loop
end = np.roll(loop, -1, axis=0)
dl = end - start
mid = 0.5 * (start + end)
dF = I_loop * np.cross(dl, B0[None, :])
tau_num = np.cross(mid, dF).sum(axis=0)
tau_ana = np.cross(m_check, B0)
print("net force on loop    :", dF.sum(axis=0), "(zero, as expected)")
print("torque (segment sum) :", tau_num)
print("torque (m x B0)      :", tau_ana)
print("agree:", np.allclose(tau_num, tau_ana))

theta = np.linspace(0.0, np.pi, 181)
m_mag = np.linalg.norm(m_check)
B_mag = np.linalg.norm(B0)
tau_mag = m_mag * B_mag * np.sin(theta)
energy = -m_mag * B_mag * np.cos(theta)

fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.plot(np.degrees(theta), tau_mag, label=r"$|\tau| = mB \sin\theta$")
ax.plot(np.degrees(theta), energy, label=r"$U = -mB \cos\theta$")
ax.axvline(0.0, color="gray", linewidth=0.8, linestyle=":")
ax.set_xlabel(r"tilt angle $\theta$ (deg)")
ax.set_ylabel("torque (N m) / energy (J)")
ax.set_title("Dipole in a uniform field: torque and energy vs orientation")
ax.grid(True, alpha=0.3)
ax.legend(loc="best")
fig

## Exercises

1. **Conceptual.** The electric field of a point charge falls as $1/r^2$ but the magnetic dipole field falls as $1/r^3$. Explain, in terms of the absence of magnetic monopoles and the multipole expansion, why the leading term of the magnetic far field is one power of $r$ steeper.

2. **Rotated dipole.** Repeat the first worked cell with $\mathbf{m} = m_0 \hat{\mathbf{x}}$: evaluate `eds.magnetic_dipole_field` on the $x$ axis (now the dipole axis) and on the $z$ axis (now equatorial), and verify numerically that the axial magnitude is exactly twice the equatorial magnitude at every sampled $r$.

3. **Equatorial convergence.** In the loop-vs-dipole comparison, sample instead along the loop's equatorial plane using `eds.biot_savart` with `closed=True` and points $(r, 0, 0)$ for $r$ from $1.5a$ to $20a$. Does the relative error still fall like $(a/r)^2$? Fit it with `eds.power_law_fit` and compare the fitted exponent and amplitude to the on-axis case.

4. **Helmholtz coils.** Build two identical loops of radius $a = 0.1$ m with `eds.circular_loop_curve`, centred at $z = \pm a/2$, each carrying $I = 1$ A in the same sense. Compute $\mathbf{B}$ on the axis with `eds.biot_savart` (call it once per loop and add the results) and plot $|B_z(z)|$; verify that near the midpoint the field is remarkably uniform, and compare the central value with the dipole estimate $\mu_0\, 2m/(4\pi (a/2)^3)$ for one coil.

5. **Energy landscape from forces.** For the single loop of the last worked cell, compute the potential energy $U(\theta) = -\mathbf{m}(\theta) \cdot \mathbf{B}_0$ by rotating the loop's vertices with a rotation matrix about $\hat{\mathbf{y}}$, recomputing the moment with `eds.magnetic_dipole_moment`, and checking $U(\theta)$ against $-mB\cos\theta$. At which angle is the torque largest, and how is that visible in $U(\theta)$?

6. **Solenoid as a stack of dipoles.** Build a solenoid with `eds.solenoid_curve(radius=0.05, length=0.5, turns=5, n_per_turn=40)`. Compute its total dipole moment with `eds.magnetic_dipole_moment` and its on-axis field with `eds.biot_savart`. Where does the dipole approximation with that single moment fail badly, and what simple physical quantity of the solenoid does $|\mathbf{m}|$ reproduce (hint: for $n$ turns spread over length $\ell$, $m = n I A \ell$)?